# SETU — GPU training on Kaggle

Trains the Hindi↔English student end-to-end on a free Kaggle GPU (T4/P100).

**Before running:** in the right-hand panel set **Accelerator → GPU T4 x2** and **Internet → On**.

Pipeline: data → teacher preferences → SFT + DPO → quantise → test → download.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable it in Settings')

In [ ]:
# clone your fork and install SETU + all extras
!git clone https://github.com/GeekyRiolu/SETU_v2.git
%cd SETU_v2/SETU
!pip -q install -e ".[data,teacher,prefs,quantize]"

In [ ]:
# switch to the full-size student + GPU hyperparameters, and run the teacher on GPU
!cp configs/model.gpu.yaml configs/model.yaml
!cp configs/training.gpu.yaml configs/training.yaml
!sed -i 's/device: cpu/device: cuda/' configs/teacher.yaml
!grep -E 'hidden_size|encoder_layers|epochs|batch_size|device' configs/model.yaml configs/training.yaml

In [ ]:
# 1) DATA: stream Samanantar (ungated) into clean, deduped parallel data.
#    Raise --limit for more data (needs Internet On). ~30k is a solid start.
!python -m setu.corpus.pipeline --limit 30000

In [ ]:
# 2) PREFERENCES: teacher n-best + kNN -> validated preference pairs (for DPO).
#    First run downloads the teacher (~400 MB). GPU makes this fast.
!python -m setu.preference.pipeline --max-entries 8000

In [ ]:
# 3) TRAIN: teacher BLEU ceiling, then SFT baseline -> DPO -> held-out eval.
#    device=auto -> CUDA. Scale --limit with your data.
!python scripts/train_full.py --limit 25000 --dev-size 200

In [ ]:
# 4) QUANTISE + EXPORT: ONNX -> INT8 -> INT4, benchmarked, deployed to models/
!python -m setu.quantize.pipeline --student dpo
!python -m setu.report --offline-proof

In [ ]:
# 5) TEST the trained student on real sentences (offline ONNX engine)
import sys; sys.path.insert(0, 'src')
from setu.inference.engine import InferenceEngine
eng = InferenceEngine(models_root='models')
print('using trained model:', not eng.is_stub)
for s in ['भारत एक विशाल देश है।',
          'मुझे किताबें पढ़ना पसंद है।',
          'आज मौसम अच्छा है।']:
    r = eng.translate(s, 'hi', 'en')
    print(f'{s}  ->  {r.translated_text}')

In [ ]:
# 6) DOWNLOAD the trained + quantised model (appears in the Output panel)
!cd models && zip -qr /kaggle/working/setu_model.zip hin_Deva-eng_Latn
!cp checkpoints/hin_Deva-eng_Latn/train_report.json /kaggle/working/ 2>/dev/null
print('Download /kaggle/working/setu_model.zip from the Output tab')